In [ ]:
import dustpy
import dustpy.constants as c
import astropy.constants as apc
import numpy as np
import matplotlib.pyplot as plt

import _opacity
import _Temperature 

c_light = apc.c.cgs.value
k_B = apc.k_B.cgs.value
sigma_sb = apc.sigma_sb.cgs.value



In [ ]:
###initialisation of the dustpy simulation by:
# S. M. Stammler and T. Birnstiel. DustPy: A Python Package for Dust Evolution in Protoplanetary Disks. 
# The Astrophysical Journal, 935(1):35, Aug. 2022. doi: 10.3847/1538-4357/ac7d58.
# URL https://stammler.github.io/dustpy/index.html
# (necessary because class Temperature is using values from there)###

sim = dustpy.Simulation()

#define the turbulent parameter alpha
sim.ini.gas.alpha = 1e-2

#initialize simulation framework
sim.initialize()

# if you want to use fixed mixing parameters instead of the same as sim.ini.gas.alpha, the delta parameters can be modified here:
#sim.dust.delta.rad = 0.001
#sim.dust.delta.turb = 0.001
#sim.dust.delta.vert = 0.001

sim.writer.overwrite = True
sim.gas.boundary.inner.value = 0
sim.dust.boundary.inner.value = 0

In [ ]:
###function for using the half time-step for updating###

old_preparator = sim.integrator.preparator

sim.Sigma_gas_old = sim.gas.Sigma.copy()
sim.Sigma_dust_old = sim.dust.Sigma.copy()
def remember_old_sigma(sim):
    sim.Sigma_gas_old = sim.gas.Sigma.copy()
    sim.Sigma_dust_old = sim.dust.Sigma.copy()
    old_preparator.beat(sim)

sim.integrator.preparator = remember_old_sigma
sim.Sigma_gas_old = sim.gas.Sigma.copy()


In [ ]:
###class opacity is called###
# used opacity file by: Birnstiel, T., Dullemond, C. P., Zhu, Z., et al. 2018, ApJL, 869, L45 
# https://github.com/birnstiel/dsharp_opac/blob/master/dsharp_opac/data/default_opacities_smooth.npz.

opac_file = 'default_opacities_smooth.npz'
#b = np.load(opac_file)
#print(b.files)
opac = _opacity.opacity(sim, opac_file) 

In [ ]:
###class Temperature is called to be used as an updater for sim.gas.T.updater###

temp_model = _Temperature.Temperature(opac)

T = temp_model.T_dustpy(sim)

sim.gas.T.updater.updater = temp_model.T_dustpy
sim.update()
T = np.asarray(sim.gas.T.data)
print(f'computed temperatures: {T}')

In [ ]:
###starting the DustPy simulation with the defined parameters and the updated temperature###
sim.run() 

In [ ]:
###plotting the default and the updated temperatures for personal reference###

op = np.load("mean_opacities.npz") # mean_opacities.npz is created in _opacity.py and can be modified there
Ttab = op["T"]
kapP = op["kappaP"]
kapR = op["kappaR"]
sigma_sb = apc.sigma_sb.cgs.value
gamma = 7/5
r = sim.grid.r
r_au = np.asarray(sim.grid.r) / c.au
Sigma = sim.gas.Sigma
H = sim.gas.Hp
h = H / r

R_star = sim.star.R
T_star = sim.star.T
kappaRg = 1e-3
kappaPg = 1e-3

exp = int(np.log10(sim.ini.gas.alpha))


Sigmad_old = sim.Sigma_dust_old
Sigmad_new = sim.dust.Sigma
Sigmad = 0.5 * (Sigmad_new + Sigmad_old)
Sigma_dust_tot = Sigmad.sum(axis=1)
Sigma_dust_tot = np.maximum(Sigma_dust_tot, 1e-10)
tau_R = 0.5 * Sigma_dust_tot * kapR
tau_P = 0.5 * Sigma_dust_tot * kapP

tau_eff = (3 * tau_R / 8) + np.sqrt(3) / 4
#adding the optically thin regions might cause T_eff_accr to behave odd for the outermost regions
#tau_eff = (3 * tau_R / 8) + np.sqrt(3) / 4 + 1 / (4 * tau_P) 
Theta0 = 2 * 4 * h / 7
R_rim = 1
Theta = Theta0 + 0.5 * (1 - Theta0) * (1- np.tanh((r - R_rim)/0.1))


nu = sim.gas.alpha * sim.gas.cs**2 / sim.grid.OmegaK
#nu = sim.gas.alpha * np.sqrt(gamma) * sim.gas.cs * H
Mdot = 3 * np.pi * Sigma * nu


# reference temperatures from: C. Dullemond, 2013, Theoretical Models of the Structure of Protoplanetary Disks, Les Houches
T_eff_accr = (3/(8 * np.pi * sigma_sb) * Mdot * sim.grid.OmegaK**2)**0.25 * (3/4 * tau_eff)**0.25
T_eff_accr_surf = (3/(8 * np.pi * sigma_sb) * Mdot * sim.grid.OmegaK**2)**0.25
T_eff_irr = (0.05 * sim.star.L/(4*np.pi*sigma_sb*r**2))**0.25

# reference from: E. I. Chiang and P. Goldreich. Spectral Energy Distributions of T Tauri Stars with Passive Circumstellar Disks. 
# The Astrophysical Journal, 490(1):368–376, Nov. 1997. doi: 10.1086/304869.
CG97 =  (Theta / 4)**0.25 * (R_star/r)**0.5 * T_star
# for the surface:
CG97_surf =  (Theta * kapP/(4 * kapR))**0.25 * (R_star/r)**0.5 * T_star






fig, ax3 = plt.subplots(1, 1, dpi=300)

ax3.plot(r_au, T, color = 'green', label = fr'updated Temperature ($\alpha = 10^{{{exp}}}$)')

# feel free to activate the reference plots if needed
#ax3.plot(r_au, T_eff_accr,'--', c='orange', alpha=0.5, zorder=0, label = r'$T^{accr}_{eff}$')
#ax3.plot(r_au, T_eff_accr_surf,'--', c='grey', alpha=0.5, zorder=0, label = r'$T^{accr}_{eff,surf}$')
#ax3.plot(r_au, T_eff_irr, '--', c='purple', alpha=0.3, zorder=0, label = r'$T^{irr}_{eff}$')
#ax3.plot(r_au, CG97,ls=':', c='red', alpha=0.5, zorder=0, label='CG97')
#ax3.plot(r_au, CG97_surf,ls=':', c='blue', alpha=0.5, zorder=0, label='CG97_surf')

ax3.set_xscale('log')
ax3.set_xlim(1, 900)
ax3.set_ylim(1,3000)
ax3.set_yscale('log')
ax3.grid('True', which = 'both')
ax3.set_xlabel(r'$r\,\left[\mathrm{au}\right]$')
ax3.set_ylabel(r'$T\,\left[\mathrm{K}\right]$')
ax3.set_title(r'temperature after $10^{5}$years')

plt.legend(loc = 'best')

plt.show()

In [ ]:
# function for plotting the single heating-terms embedded in _Temperature.py
temp_model.plot_heating()
# function for plotting the opacities embedded in _opacity.py in 2d and 3d
opac.plot_kappa()


In [ ]:
###default DustPy ipanel plots###

dustpy.plot.ipanel("data", it=10)
dustpy.plot.ipanel("data", it=15)
dustpy.plot.ipanel("data", it=21)